# Risk Parameters of BTC EV Fiat Market

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
pd.options.plotting.backend = "plotly"

import funding
import impact
import liquidations as liq
import pricedrift as drift
import pystable
from tqdm import tqdm

Set data-specific parameters:

In [3]:
file_name = "/Users/fredericoteixeira/Projects/overlay/data/btc_ev_v12_fiat.csv"
periodicity = 1.0  # 1 day in seconds
cap = 10  # cap on pay off, set by governance

## Understanding the Data

Load and plot the data:

In [4]:
df = pd.read_csv(file_name).set_index("date").drop("created_at", axis=1)
df.plot()

Compute `k`, the funding related risk metric:

In [8]:
ks, dst = funding.generic_get_ks(df["btc_ev_v12_fiat_index"].to_numpy(), periodicity)
df_ks = pd.DataFrame(
    data=ks,
    columns=[alpha for alpha in funding.ALPHAS],
    index=[n/funding.NS[0] for n in funding.NS]
)
df_ks.columns.name = "alpha"
df_ks.index.name = "days"

df_ks.plot()

Draw the histogram of log returns of the index (this will be compared against the probability distribution later)

In [ ]:
log_diff = np.log(df["btc_ev_v12_fiat_index"].div(df["btc_ev_v12_fiat_index"].shift(1)).dropna())
fig = px.histogram(log_diff, nbins=100)
fig.show()

Plot the histogram of the fitted Stable distribution:

In [ ]:
draws = pystable.rnd(dst, n=len(log_diff))
fig = px.histogram(np.sort(np.clip(draws, -0.2, 0.2)), nbins=100)
fig.show()

## Funding Rate parameters

What if we had started trading this index `3` years ago, and using all data available to compute the index? 

We look back the whole history of data for each day and compute the risk parameters.

For simplicity, we just collect results for $1$ day prediction.

In [ ]:
rolling_window = 3 * 365  # 3 years
prediction = 0  # 1 day prediction is the 0-th row of the df_ks dataframe

In [ ]:
df_ks_hist = pd.DataFrame(
    0.,
    columns=df_ks.columns,
    index=df.index[rolling_window:]
)
df_ks_hist.columns.name = df_ks.columns.name
df_ks_hist.index.name = "date"

In [ ]:
df_dst_hist = pd.DataFrame(
    0.,
    columns=["alpha", "beta", "mu", "sigma"],
    index=df.index[rolling_window:]
)
df_dst_hist.columns.name = "parameters"
df_dst_hist.index.name = "date"

In [ ]:
bar = tqdm(total=df_ks_hist.index.size)

for today in df_ks_hist.index:

    tmp_df_ks, tmp_dst = funding.get_ks(df.loc[:today, "btc_ev_v12_fiat_index"], periodicity=1)
    
    df_ks_hist.loc[today, :] = tmp_df_ks.iloc[prediction, :].to_numpy()
    df_dst_hist.loc[today, :] = np.array(
        [dst.contents.alpha, dst.contents.beta, dst.contents.mu_1, dst.contents.sigma]
    )

    bar.update(1)

bar.close()

In [ ]:
df_ks_hist.plot(title="Funding rate parameters")

In [ ]:
df_ks_hist.div(df_ks_hist.iloc[0,:]).plot(title="Normalized funding rate parameters")

In [ ]:
def plot_df_ks_hist(alpha):
    return df_ks_hist.loc[:,alpha].plot(title=f"Funding rate with alpha = {alpha}")